In [82]:
!pip install requests pandas numpy tqdm -q

import requests
import pandas as pd
import numpy as np
import ast
import time
import json
from tqdm import tqdm

In [83]:
# Файл с локациями на GitHub
GITHUB_CSV = (
    "https://raw.githubusercontent.com/"
    "Unchell57/CoffeePlusPlus/main/output/locations.csv"
)

# Ключ 2GIS
DGIS_KEY = "0d57ee04-d636-43c5-821b-e9118c9fb476"

# Радиус поиска вокруг каждой локации
RADIUS = 500   # метров

# Задержка между запросами- чтобы не превысить лимит API (Вроде ограничение 1000)
DELAY = 0.5    # секунд

# Количество результатов на один запрос к 2GIS
PAGE_SIZE = 10


In [84]:
DGIS_URL = "https://catalog.api.2gis.com/3.0/items"

def dgis_search(lat: float, lon: float, query: str) -> list:
    """
    Ищет объекты через 2GIS Catalog API v3.

    Параметры:
      lat, lon- координаты центра поиска
      query- текстовый запрос, например 'кофейня' или 'университет'
    Возвращает список объектов (dict) или [] при ошибке.

    2GIS принимает координаты в порядке lon,lat
    """
    params = {
        "q":         query,
        "point":     f"{lon},{lat}",      # ← порядок: долгота, широта
        "radius":    RADIUS,
        "fields":    "items.point,items.rubrics,items.price_comment,"
                     "items.reviews,items.name",
        "key":       DGIS_KEY,
        "page_size": PAGE_SIZE,
        "locale":    "ru_RU",
    }
    try:
        resp = requests.get(DGIS_URL, params=params, timeout=15)

        # Если сервер вернул ошибку возвращает пустой список
        if resp.status_code != 200:
            return []

        data = resp.json()
        return data.get("result", {}).get("items", [])

    except Exception:
        return []

In [85]:
import re

# Таблица: ключевое слово → средний чек (руб.)
PRICE_KEYWORDS = {
    "очень дёшево":  150,
    "дёшево":        200,
    "недорого":      250,
    "до 300":        250,
    "до 500":        350,
    "средне":        450,
    "выше среднего": 650,
    "дорого":        900,
    "очень дорого": 1500,
}

# Известные сети и их средний чек по открытым данным
CHAIN_CHECKS = {
    "starbucks": 550, "старбакс": 550,
    "cofix":     200, "кофикс":   200,
    "surf":      450,
    "double b":  450, "даблби":   450,
    "black star":400,
    "шоколадница": 350,
    "coffee house": 320, "кофе хауз": 320,
    "coffee like":  220, "кофе лайк": 220,
    "traveler":  380,
    "prime":     200, "прайм":    200,
    "бодрый":    190,
    "mcdonalds": 180, "макдо":    180,
}

DEFAULT_CHECK = 320   # среднее по московским кофейням

def parse_check_from_comment(comment: str) -> int:
    """
    Парсит средний чек из текстового поля price_comment.
    Сначала ищет число, потом ключевые слова.
    """
    if not comment:
        return 0

    text = comment.lower()

    # ищет число
    numbers = re.findall(r'\d+', text)
    if numbers:
        val = int(numbers[0])
        if 100 <= val <= 5000:   # разумный диапазон для чека
            return val

    # ищет ключевые слова
    for kw, check in PRICE_KEYWORDS.items():
        if kw in text:
            return check

    return 0


def get_avg_check(items: list) -> int:
    """
    Считает средний чек по списку кофеен из 2GIS.
    Для каждой позиции пробует: Поле price_comment, название (совпадение с известной сетью), DEFAULT_CHECK
    """
    if not items:
        return 0

    checks = []
    for item in items:
        # поле price_comment от 2GIS
        pc    = item.get("price_comment") or ""
        check = parse_check_from_comment(pc)

        # имя совпадает с известной сетью
        if not check:
            name = (item.get("name") or "").lower()
            for chain, val in CHAIN_CHECKS.items():
                if chain in name:
                    check = val
                    break

        # дефолт
        if not check:
            check = DEFAULT_CHECK

        checks.append(check)

    return int(round(np.mean(checks)))

In [86]:
def collect_one(row: pd.Series) -> dict:
    lat = row["lat"]
    lon = row["lon"]

    # ── Кофейни: один простой запрос ───────────────────────────
    cafes = dgis_search(lat, lon, "кофейня")
    time.sleep(DELAY)

    # ── Образование: три отдельных запроса + объединяем ────────
    # (один запрос "университет институт вуз" 2GIS не понимает)
    univs = []
    for q in ["университет", "институт", "академия"]:
        univs += dgis_search(lat, lon, q)
        time.sleep(DELAY)

    # Убираем дубликаты по id
    seen = set()
    univs_unique = []
    for u in univs:
        if u.get("id") not in seen:
            seen.add(u.get("id"))
            univs_unique.append(u)

    # ── IT и офисы: отдельные запросы ──────────────────────────
    it_offices = []
    for q in ["IT компания", "бизнес-центр", "коворкинг"]:
        it_offices += dgis_search(lat, lon, q)
        time.sleep(DELAY)

    seen2 = set()
    it_unique = []
    for o in it_offices:
        if o.get("id") not in seen2:
            seen2.add(o.get("id"))
            it_unique.append(o)

    cafe_count = len(cafes)
    avg_check  = get_avg_check(cafes)
    univ_count = len(univs_unique)
    it_count   = len(it_unique)
    attractor  = univ_count * 3 + it_count * 2

    return {
        "cafe_count": cafe_count,
        "avg_check":  avg_check,
        "univ_count": univ_count,
        "it_count":   it_count,
        "attractor":  attractor,
    }

In [87]:
def collect_all(df: pd.DataFrame) -> pd.DataFrame:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    WORKERS = 4
    rows  = [None] * len(df)
    items = list(df.iterrows())
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = {
            pool.submit(collect_one, row): idx
            for idx, (_, row) in enumerate(items)
        }
        pbar = tqdm(as_completed(futures), total=len(futures), desc="2GIS запросы")
        for future in pbar:
            idx       = futures[future]
            rows[idx] = future.result()
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

In [88]:
# Оценка от 0 до 100.
#  Цена/м² (инверс.)   25%    Ниже аренда → выше балл
#  Точки притяжения ЦА 20%    Вузы (×3) + IT-офисы (×2)
#  Средний чек района  15%    Выше чек → платёжеспособнее район
#  Время до метро      15%    Ближе → доступнее для клиентов
#  Вышки рядом         10%    Прокси проходимости места
#  Кофейни рядом       10%    Активный район (clip 15 — не штраф)
#  Площадь (sweet spot) 5%    Оптимум 25–80 м² для кофейни

WEIGHTS = {
    "price_per_sqm": 0.25,
    "attractor":     0.20,
    "avg_check":     0.15,
    "metro_min":     0.15,
    "towers_nearby": 0.10,
    "cafe_count":    0.10,
    "area_score":    0.05,
}


def minmax(s: pd.Series) -> pd.Series:
    """Нормализует ряд к диапазону 0–1."""
    lo, hi = s.min(), s.max()
    return pd.Series(0.5, index=s.index) if hi == lo else (s - lo) / (hi - lo)


def area_score(a: float) -> float:
    """
    Sweet-spot оценка площади:
    Слишком маленькая (<15 м²) — не развернуть кофейню.
    Слишком большая (>120 м²) — дорого и сложно наполнить.
    Оптимум 25–80 м² → 1.0
    """
    if   a < 10:   return 0.0
    elif a < 20:   return 0.3
    elif a < 25:   return 0.6
    elif a <= 80:  return 1.0   # идеальный диапазон
    elif a <= 110: return 0.75
    elif a <= 130: return 0.5
    else:          return 0.25


def compute_scores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Вычисляет итоговый балл для каждой локации: нормализует каждый фактор к [0, 1], Для «чем меньше- тем лучше» берём инверсию: 1- x, складывает с весами- результат умножаем на 100
    """
    d = df.copy()

    # Цена/м²: ниже = лучше → инвертирует
    d["f_price"]     = 1 - minmax(d["price_per_sqm"])

    # Притяжение ЦА: больше = лучше
    d["f_attractor"] = minmax(d["attractor"])

    # Средний чек: если 2GIS ничего не вернул — нейтральное 0.5
    if d["avg_check"].sum() == 0:
        d["f_check"] = 0.5
    else:
        d["f_check"] = minmax(d["avg_check"])

    # Метро: меньше минут = лучше → инвертирует
    d["f_metro"]     = 1 - minmax(d["metro_min"])

    # Вышки: больше = лучше (больше людей с телефонами = больше проходимость)
    d["f_towers"]    = minmax(d["towers_nearby"])

    # Кофейни: обрезаем на 15 (выше — уже перенасыщение)
    d["f_cafes"]     = minmax(d["cafe_count"].clip(0, 15))

    # Площадь: sweet spot
    d["f_area"]      = d["area"].apply(area_score)

    # Итоговый балл: взвешенная сумма × 100
    d["score"] = (
        d["f_price"]     * WEIGHTS["price_per_sqm"] +
        d["f_attractor"] * WEIGHTS["attractor"]     +
        d["f_check"]     * WEIGHTS["avg_check"]     +
        d["f_metro"]     * WEIGHTS["metro_min"]     +
        d["f_towers"]    * WEIGHTS["towers_nearby"] +
        d["f_cafes"]     * WEIGHTS["cafe_count"]    +
        d["f_area"]      * WEIGHTS["area_score"]
    ) * 100

    d["score"] = d["score"].round(1)
    return d


def top10(df: pd.DataFrame) -> pd.DataFrame:
    """Возвращает топ-10 локаций с нужными колонками."""
    t = df.sort_values("score", ascending=False).head(10).copy()
    t.insert(0, "rank", range(1, 11))
    return t[[
        "rank", "address", "metro", "metro_time",
        "price", "area", "price_per_sqm",
        "cafe_count", "avg_check",
        "univ_count", "it_count", "attractor",
        "towers_nearby", "score",
    ]].reset_index(drop=True)

In [89]:
def print_results(t: pd.DataFrame, total: int) -> None:
    """печатает топ-10 в консоль."""
    for _, r in t.iterrows():
        print(f"\n  #{int(r['rank'])}  {r['address']}")
        print(f"      Метро: {r['metro']} ({r['metro_time']})")
        print(f"      Аренда: {int(r['price']):>7,} ₽/мес  "
              f"│  Площадь: {r['area']:>6} м²  "
              f"│  {int(r['price_per_sqm']):>5,} ₽/м²")
        print(f"      Кофейни рядом: {int(r['cafe_count'])}  "
              f"│  Ср. чек: {int(r['avg_check'])} ₽  "
              f"│  Вышки: {int(r['towers_nearby'])}")
        print(f"      ЦА: вузов {int(r['univ_count'])}  "
              f"│  IT-офисов {int(r['it_count'])}  "
              f"│  Балл притяжения: {int(r['attractor'])}")
        print(f"      {' ИТОГОВЫЙ БАЛЛ':>20}: {r['score']} / 100")

    labels = {
        "price_per_sqm": "Цена аренды (инверс.)",
        "attractor":     "Точки притяжения ЦА",
        "avg_check":     "Средний чек района",
        "metro_min":     "Близость метро (инверс.)",
        "towers_nearby": "Вышки / проходимость",
        "cafe_count":    "Кофейни рядом",
        "area_score":    "Площадь (sweet-spot)",
    }
    for key, w_val in WEIGHTS.items():
        bar  = "█" * int(w_val * 50)
        print(f"  {labels[key]:<30}  {int(w_val*100):2d}%  {bar}")

    print(f"\n  Всего проанализировано: {total} локаций")


def save_and_download(df: pd.DataFrame, t: pd.DataFrame) -> None:
    """Сохраняет результаты в CSV и скачивает из Colab."""
    full_path = "/content/locations_scored.csv"
    top_path  = "/content/top10_locations.csv"

    # Сохраняю полный список с баллами
    df.to_csv(full_path, index=False, encoding="utf-8-sig")

    # Сохраняю топ-10
    t.to_csv(top_path, index=False, encoding="utf-8-sig")

    try:
        from google.colab import files
        files.download(full_path)
        files.download(top_path)
    except ImportError:
        pass

In [90]:
import pandas as pd, ast, io, urllib.request

def load_locations():

    # Читаем CSV напрямую по URL
    response = urllib.request.urlopen(GITHUB_CSV)
    df = pd.read_csv(io.BytesIO(response.read()))

    # Координаты из строки {'lat':...,'lon':...}
    df["lat"] = df["cords"].apply(lambda s: ast.literal_eval(s)["lat"])
    df["lon"] = df["cords"].apply(lambda s: ast.literal_eval(s)["lon"])

    # Время до метро в минуты
    metro_to_min = {
        "до 5 мин.":    3,
        "6–10 мин.":    8,
        "11–15 мин.":  13,
        "16–20 мин.":  18,
        "21–30 мин.":  25,
        "от 31 мин.":  40,
    }
    df["metro_min"] = df["metro_time"].map(metro_to_min).fillna(25).astype(int)
    df["price_per_sqm"] = (df["price"] / df["area"]).round(0).astype(int)

    return df

In [91]:
def main():
    #загрузка
    df = load_locations()

    # запросы к 2GIS
    est_min = len(df) * DELAY * 3 / 60
    df = collect_all(df)

    # метрика
    df = compute_scores(df)

    #результаты
    t = top10(df)
    print_results(t, len(df))
    save_and_download(df, t)

    return df, t

In [92]:
df_result, top_result = main()

2GIS запросы: 100%|██████████| 343/343 [09:44<00:00,  1.70s/it]


  #1  Пятницкая ул., 37
      Метро: Третьяковская (до 5 мин.)
      Аренда:  60,000 ₽/мес  │  Площадь:   60.0 м²  │  1,000 ₽/м²
      Кофейни рядом: 10  │  Ср. чек: 346 ₽  │  Вышки: 17
      ЦА: вузов 16  │  IT-офисов 25  │  Балл притяжения: 98
             ИТОГОВЫЙ БАЛЛ: 90.8 / 100

  #2  Арбатский пер., 2/6
      Метро: Арбатская (до 5 мин.)
      Аренда:  10,000 ₽/мес  │  Площадь:   50.0 м²  │    200 ₽/м²
      Кофейни рядом: 10  │  Ср. чек: 336 ₽  │  Вышки: 15
      ЦА: вузов 14  │  IT-офисов 24  │  Балл притяжения: 90
             ИТОГОВЫЙ БАЛЛ: 89.0 / 100

  #3  Большая Садовая ул., 5к1
      Метро: Маяковская (до 5 мин.)
      Аренда: 200,000 ₽/мес  │  Площадь:   58.0 м²  │  3,448 ₽/м²
      Кофейни рядом: 10  │  Ср. чек: 333 ₽  │  Вышки: 25
      ЦА: вузов 5  │  IT-офисов 30  │  Балл притяжения: 75
             ИТОГОВЫЙ БАЛЛ: 86.3 / 100

  #4  Новослободская ул., 20
      Метро: Менделеевская (до 5 мин.)
      Аренда:  25,000 ₽/мес  │  Площадь:   11.3 м²  │  2,212 ₽/м²
      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>